# Step 2: Data understanding and cleaning

**Input:** `data/raw/city_day.csv` (Kaggle "Air Quality Data in India (2015-2020)", source CPCB).
**Output:** `data/interim/city_day_checked.csv` and tables in `report/tables/`, figures in `report/figures/`.

Rules for this notebook:
* **No imputation here.** Imputation is fitted on the training years only, in Step 4. Here we only *document* missingness.
* **No outlier deletion.** Outliers are flagged, not removed, unless physically impossible (negative concentrations).
* Every drop decision is printed with its rule, so it can be quoted in the report.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "city_day.csv"
INTERIM = ROOT / "data" / "interim"
FIG = ROOT / "report" / "figures"
TAB = ROOT / "report" / "tables"
for d in (INTERIM, FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

POLLUTANTS = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3",
              "Benzene", "Toluene", "Xylene"]
BUCKET_ORDER = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
pd.set_option("display.width", 160)

## 2.1 Load, shape, dtypes, date column

In [ ]:
df = pd.read_csv(RAW)
print("shape:", df.shape)
print(df.dtypes)
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d", errors="coerce")
print("unparseable dates:", df["Date"].isna().sum())
df = df.sort_values(["City", "Date"]).reset_index(drop=True)
df.head()

### Facts to record in the report (Step 1 also asks for these)

In [ ]:
facts = {
    "rows": len(df),
    "cities": df["City"].nunique(),
    "date_min": df["Date"].min().date(),
    "date_max": df["Date"].max().date(),
    "columns": list(df.columns),
}
for k, v in facts.items():
    print(f"{k}: {v}")

city_span = (df.groupby("City")["Date"]
               .agg(first="min", last="max", n_rows="size")
               .sort_values("first"))
city_span["expected_days"] = (city_span["last"] - city_span["first"]).dt.days + 1
city_span["missing_dates"] = city_span["expected_days"] - city_span["n_rows"]
city_span.to_csv(TAB / "city_reporting_span.csv")
city_span

**Why `missing_dates` matters:** in Step 4 the lag features (`yesterday's PM2.5`) and the target
(`tomorrow is unhealthy`) are built with a shift. If a city has gaps in its dates, a row-based
`shift(1)` silently pairs non-consecutive days. If this column is not all zeros, Step 4 must
reindex each city to a full daily calendar before shifting.

## 2.2 Missing values per column and per city

In [ ]:
miss_col = (df[POLLUTANTS + ["AQI", "AQI_Bucket"]].isna().mean()
              .mul(100).round(1).sort_values(ascending=False)
              .rename("pct_missing").to_frame())
miss_col.to_csv(TAB / "missing_by_column.csv")
miss_col

In [ ]:
miss_city = (df.groupby("City")[POLLUTANTS + ["AQI"]]
               .apply(lambda g: g.isna().mean() * 100).round(1))
miss_city = miss_city.loc[miss_city.mean(axis=1).sort_values().index]
miss_city.to_csv(TAB / "missing_by_city_column.csv")

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(miss_city, cmap="Reds", vmin=0, vmax=100, annot=True, fmt=".0f",
            annot_kws={"size": 7}, cbar_kws={"label": "% missing"}, ax=ax)
ax.set_title("Percent missing by city and pollutant (city_day.csv)")
plt.tight_layout(); plt.savefig(FIG / "missing_heatmap_city.png", dpi=150); plt.show()

Note: a 100% cell means that city **never measured** that pollutant (no sensor). That is
different from random gaps and cannot be imputed meaningfully from the city's own history.
Say this in the report.

## 2.3 Duplicates (same city and date twice)

In [ ]:
dup_mask = df.duplicated(subset=["City", "Date"], keep=False)
print("rows involved in City+Date duplicates:", dup_mask.sum())
print("fully identical rows:", df.duplicated(keep=False).sum())
df[dup_mask].head(10)

In [ ]:
# Policy: exact copies are dropped. Conflicting duplicates (same key, different values) are
# printed above and must be inspected by hand before choosing a rule.
n_before = len(df)
df = df.drop_duplicates()
print("dropped exact duplicate rows:", n_before - len(df))
assert not df.duplicated(subset=["City", "Date"]).any(), "conflicting duplicates remain: inspect them"

## 2.4 Impossible values: negatives

In [ ]:
neg = (df[POLLUTANTS + ["AQI"]] < 0).sum()
print("negative values per column:\n", neg[neg > 0] if neg.any() else "none")
zeros = (df[POLLUTANTS] == 0).sum()
print("\nexact zeros per column (suspicious for some sensors, not impossible):\n", zeros[zeros > 0])

In [ ]:
# A negative concentration is physically impossible: treat as missing (not as 0).
df[POLLUTANTS] = df[POLLUTANTS].mask(df[POLLUTANTS] < 0)

## 2.5 Outliers: boxplots and IQR flags

Two design choices, both deliberate:

1. **Log scale.** Pollutants are strongly right-skewed. The IQR rule on raw values flags a large
   share of legitimate high-pollution days. Applying it to `log1p(x)` flags values that are
   extreme *relative to a skewed distribution*.
2. **Per city.** Pooled across cities, Delhi's normal winter days look like outliers compared
   with Bengaluru. Flagging within each city asks "is this value unusual *for this city*?".

Units: CO is in mg/m³, the rest in µg/m³, so do not compare boxes across pollutants by height.

In [ ]:
long = df.melt(id_vars=["City", "Date"], value_vars=POLLUTANTS,
               var_name="pollutant", value_name="value").dropna()
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=long, x="pollutant", y=np.log1p(long["value"]), ax=ax, fliersize=1)
ax.set_ylabel("log1p(concentration)"); ax.set_title("Pollutant distributions (log1p scale, all cities)")
plt.tight_layout(); plt.savefig(FIG / "boxplot_pollutants_log.png", dpi=150); plt.show()

In [ ]:
def iqr_flags(s, k=1.5):
    x = np.log1p(s)
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (x < q1 - k * iqr) | (x > q3 + k * iqr)

flags = pd.DataFrame(index=df.index)
for p in POLLUTANTS:
    flags[p] = df.groupby("City")[p].transform(iqr_flags).fillna(False).astype(bool)

out_summary = pd.DataFrame({
    "n_flagged": flags.sum(),
    "pct_of_observed": (flags.sum() / df[POLLUTANTS].notna().sum() * 100).round(2),
})
out_summary.to_csv(TAB / "iqr_outliers_by_pollutant.csv")
out_summary

### Are the extremes real events? Check the month distribution

If high-side flags cluster in Oct-Nov (Diwali, stubble burning) and Dec-Jan (winter inversion)
in northern cities, they are real pollution episodes and must stay in the data: they are exactly
the unhealthy days RQ2 tries to predict. Isolated single-day spikes in summer, or values that jump
by orders of magnitude and drop back the next day, are more likely sensor errors.

In [ ]:
high = pd.DataFrame(index=df.index)
for p in POLLUTANTS:
    x = np.log1p(df[p])
    med = x.groupby(df["City"]).transform("median")
    high[p] = flags[p] & (x > med)   # keep only high-side flags

df["month"] = df["Date"].dt.month
pm_high = df.loc[high["PM2.5"], ["City", "month"]]
tab = pd.crosstab(pm_high["City"], pm_high["month"])
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(tab, cmap="Blues", annot=True, fmt="d", annot_kws={"size": 7}, ax=ax)
ax.set_title("High-side PM2.5 IQR flags (per-city, log scale) by month")
plt.tight_layout(); plt.savefig(FIG / "pm25_outliers_by_month.png", dpi=150); plt.show()

In [ ]:
for city in ["Delhi", "Lucknow", "Patna", "Gurugram"]:
    if city not in df["City"].unique():
        continue
    m = high.loc[df["City"] == city, "PM2.5"]
    months = df.loc[m[m].index, "month"]
    share = months.isin([10, 11, 12, 1]).mean() if len(months) else float("nan")
    print(f"{city}: {len(months)} high PM2.5 flags, {share:.0%} fall in Oct-Jan")

In [ ]:
# Isolated spikes: value > 5x both neighbours on consecutive days. Candidates for sensor error.
def isolated_spikes(g, p, ratio=5):
    g = g.set_index("Date").asfreq("D")
    s = g[p]
    prev, nxt = s.shift(1), s.shift(-1)
    hit = (s > ratio * prev) & (s > ratio * nxt) & (s > s.quantile(0.9))
    return g.loc[hit, [p]].assign(prev=prev[hit], next=nxt[hit])

spikes = {p: df.groupby("City")[["Date", p]].apply(isolated_spikes, p=p) for p in ["PM2.5", "PM10", "CO"]}
for p, t in spikes.items():
    print(f"{p}: {len(t)} isolated spikes")
spikes["PM2.5"].head(15)

**Decision for this notebook:** keep all positive values. Save the outlier and spike flags as
columns so later steps can (a) run a robustness check with flagged spikes excluded, and
(b) check in Step 6 whether models fail more on spikes.

In [ ]:
df["any_iqr_flag"] = flags.any(axis=1)
spike_idx = set()
for p, t in spikes.items():
    for (city, date) in t.index:
        spike_idx.add((city, date))
df["isolated_spike"] = [ (c, d) in spike_idx for c, d in zip(df["City"], df["Date"]) ]
df[["any_iqr_flag", "isolated_spike"]].mean()

## 2.6 AQI vs AQI_Bucket consistency

Note on the plan: a missing **bucket** can be recovered from AQI (deterministic). A missing
**AQI** cannot be recovered from the bucket, only bounded to a range. So the repair goes one way.

In [ ]:
def aqi_to_bucket(a):
    return pd.cut(a, bins=[-np.inf, 50, 100, 200, 300, 400, np.inf],
                  labels=BUCKET_ORDER, right=True)

df["bucket_from_aqi"] = aqi_to_bucket(df["AQI"])
both = df["AQI"].notna() & df["AQI_Bucket"].notna()
mismatch = both & (df["bucket_from_aqi"].astype(str) != df["AQI_Bucket"].astype(str))
print("rows with both AQI and bucket:", both.sum())
print("mismatches:", mismatch.sum())
print("AQI present, bucket missing:", (df["AQI"].notna() & df["AQI_Bucket"].isna()).sum())
print("bucket present, AQI missing:", (df["AQI"].isna() & df["AQI_Bucket"].notna()).sum())
print("both missing:", (df["AQI"].isna() & df["AQI_Bucket"].isna()).sum())
pd.crosstab(df.loc[both, "AQI_Bucket"], df.loc[both, "bucket_from_aqi"])

In [ ]:
# Mismatches near a boundary (e.g. AQI 100.4) are rounding; others need a look.
df.loc[mismatch, ["City", "Date", "AQI", "AQI_Bucket", "bucket_from_aqi"]].head(20)

In [ ]:
# Repair: fill missing bucket from AQI. Use the recomputed bucket as the canonical label,
# so the RQ2 target follows exactly the thresholds stated in the report.
df["AQI_Bucket"] = df["bucket_from_aqi"].astype("object").where(df["AQI"].notna(), df["AQI_Bucket"])
df = df.drop(columns="bucket_from_aqi")

## 2.7 Drop columns or cities that are too sparse

Thresholds are parameters. Pick them after reading the tables above, then **write the rule and
the reason in the report**. Suggested starting point:

* **Columns:** drop a pollutant if > 50% missing overall. Reason: imputing most of a column
  means the model learns the imputer, not the data.
* **Cities:** drop a city if AQI is missing on > 50% of its days, because AQI is the target for
  both RQ1 and RQ2.

Missing AQI rows are **not** dropped here: they still carry pollutant readings that lag features
in Step 4 can use. Drop them only where a target is needed.

In [ ]:
COL_THRESH = 50   # % missing
CITY_AQI_THRESH = 50

drop_cols = miss_col.index[(miss_col["pct_missing"] > COL_THRESH) & miss_col.index.isin(POLLUTANTS)].tolist()
city_aqi_miss = df.groupby("City")["AQI"].apply(lambda s: s.isna().mean() * 100)
drop_cities = city_aqi_miss[city_aqi_miss > CITY_AQI_THRESH].index.tolist()
print(f"columns > {COL_THRESH}% missing -> drop:", drop_cols)
print(f"cities with AQI > {CITY_AQI_THRESH}% missing -> drop:", drop_cities)
print(city_aqi_miss.round(1).sort_values(ascending=False).head(8))

In [ ]:
clean = df.drop(columns=drop_cols)
clean = clean[~clean["City"].isin(drop_cities)].drop(columns="month")
print("before:", df.shape, " after:", clean.shape, " cities:", clean["City"].nunique())
clean.to_csv(INTERIM / "city_day_checked.csv", index=False)

## 2.8 Summary for the report (fill in from the outputs above)

| Check | Result | Action |
|---|---|---|
| Rows / cities / date range | | |
| Date gaps per city | | reindex to daily before lags (Step 4) if non-zero |
| Columns > 50% missing | | dropped |
| Cities dropped | | |
| Duplicates | | |
| Negative values | | set to missing |
| IQR outliers (per-city, log) | | kept, flagged |
| Do PM2.5 extremes cluster in Oct-Jan in northern cities? | | |
| Isolated spikes | | kept, flagged for robustness check |
| AQI vs bucket mismatches | | bucket recomputed from AQI |